# Model Inference — submission

In [ ]:
%pip install -q mlflow pandas matplotlib scikit-learn

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    pass
sys.path.insert(0, os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow

from src.data import load_raw, submission_ids
from src.features import WalmartFeatureBuilder, MARKDOWN_COLS
from src.metrics import wmae, holiday_weights
from src.validation import year_back_split, time_folds, DEV_TRAIN_END, TRAIN_END
from src.tracking import setup_mlflow, REGISTERED_MODEL_NAME

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("MLflow tracking:", setup_mlflow("Model_Inference"))

In [ ]:
MODEL_URI = f"models:/{REGISTERED_MODEL_NAME}@champion"
# champion registry-დან
model = mlflow.pyfunc.load_model(MODEL_URI)
print("ჩაიტვირთა:", MODEL_URI)
print("წყარო run:", model.metadata.run_id)

In [ ]:
raw = load_raw()
test = raw["test"]

pred = model.predict(test)
assert len(pred) == len(test) == 115064

submission = pd.DataFrame({"Id": submission_ids(test), "Weekly_Sales": pred})
submission.to_csv("submission.csv", index=False)
print(f"შენახულია: submission.csv")
submission.head()

In [ ]:
train = raw["train"]
hist = train.groupby("Date")["Weekly_Sales"].sum() / 1e6
fc = submission.assign(Date=pd.to_datetime(test["Date"].values)) \
               .groupby("Date")["Weekly_Sales"].sum() / 1e6

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(hist.index[-60:], hist.values[-60:], label="history (train)")
ax.plot(fc.index, fc.values, color="tomato", label="prediction (test)")
for d in pd.to_datetime(["2012-11-23", "2012-12-28", "2013-02-08"]):
    ax.axvline(d, color="gray", ls=":", lw=1)
ax.set_title("Total weekly sales, $M  (dotted: Thanksgiving / Christmas / Super Bowl)")
ax.legend(); plt.tight_layout(); plt.show()

print(f"პროგნოზის სტატისტიკა: min={pred.min():.0f}, mean={pred.mean():.0f}, "
      f"max={pred.max():.0f}, უარყოფითი: {(pred < 0).mean():.2%}")